In [1]:
pip install mp-api pymatgen

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 2.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.4/119.4 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 86.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 308.8/308.8 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.9/51.9 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.3/332.3 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 962.5/962.5 kB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 95.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.4/127.4 kB 10.5 MB/s eta 0:00:00
  

In [ ]:
import os

# Replace 'YOUR_API_KEY_HERE' with your actual Materials Project API key.
os.environ["MP_API_KEY"] = "DuM5BwtimSdVR2d4I3D6HGtu07xOkrKU"

In [3]:
import pandas as pd
import numpy as np
from pymatgen.core import Composition, Element
from mp_api.client import MPRester
from tqdm import tqdm
import os
import warnings

warnings.filterwarnings("ignore")

# -------------------------------------------------------------------
# CONFIG
# -------------------------------------------------------------------
os.environ["MP_API_KEY"] = "DuM5BwtimSdVR2d4I3D6HGtu07xOkrKU"
API_KEY = os.environ["MP_API_KEY"]

VPA_MIN, VPA_MAX = 5, 100   # physical filter

# -------------------------------------------------------------------
# HELPER: composition → elements & fractions
# -------------------------------------------------------------------
def extract_elements_and_fractions(formula):
    comp = Composition(formula)
    el_amt = comp.get_el_amt_dict()
    total = sum(el_amt.values())
    return (
        list(el_amt.keys()),
        [v / total for v in el_amt.values()]
    )

# -------------------------------------------------------------------
# HELPER: physics-based composition features
# -------------------------------------------------------------------
def composition_physics_features(formula):
    comp = Composition(formula)
    total = comp.num_atoms

    radii, ens, vals, masses, fracs = [], [], [], [], []

    for el, amt in comp.items():
        frac = amt / total
        fracs.append(frac)

        radii.append((el.atomic_radius or el.atomic_radius_calculated or np.nan))
        ens.append(el.X if el.X is not None else np.nan)
        vals.append(el.group if el.group is not None else np.nan)
        masses.append(float(el.atomic_mass))

    fracs = np.array(fracs)

    def wmean(x):
        x = np.array(x, dtype=float)
        mask = ~np.isnan(x)
        return np.sum(x[mask] * fracs[mask]) if mask.any() else np.nan

    def wstd(x):
        x = np.array(x, dtype=float)
        mask = ~np.isnan(x)
        if not mask.any():
            return np.nan
        mean = np.sum(x[mask] * fracs[mask])
        return np.sqrt(np.sum(fracs[mask] * (x[mask] - mean) ** 2))

    return {
        "mean_atomic_radius": wmean(radii),
        "std_atomic_radius": wstd(radii),
        "mean_electronegativity": wmean(ens),
        "mean_valence_electrons": wmean(vals),
        "mean_atomic_mass": wmean(masses),
    }

# -------------------------------------------------------------------
# STEP 1: FETCH RAW MP DATA
# -------------------------------------------------------------------
records = []

with MPRester(API_KEY) as mpr:
    entries = mpr.summary.search(
        nsites=(1, 200),
        fields=["material_id", "formula_pretty", "structure"]
    )

    for e in tqdm(entries, desc="Fetching MP data"):
        try:
            struct = e.structure
            if struct is None:
                continue

            num_atoms = len(struct)                 # ✅ CORRECT
            total_volume = struct.volume
            vpa = total_volume / num_atoms

            if not (VPA_MIN <= vpa <= VPA_MAX):
                continue

            records.append({
                "material_id": str(e.material_id),
                "composition": e.formula_pretty,
                "num_atoms": num_atoms,
                "total_volume": total_volume,
                "volume_per_atom": vpa,
            })

        except Exception:
            continue

raw_df = pd.DataFrame(records)
print("Raw dataset:", raw_df.shape)

# -------------------------------------------------------------------
# STEP 2: DEDUPLICATE BY COMPOSITION (CRITICAL FIX)
# -------------------------------------------------------------------
# Aggregate polymorphs safely
df = (
    raw_df
    .groupby("composition", as_index=False)
    .agg({
        "volume_per_atom": "mean",   # or "median"
    })
)

print("After composition deduplication:", df.shape)

# -------------------------------------------------------------------
# STEP 3: ADD ELEMENTS, FRACTIONS, PHYSICS FEATURES
# -------------------------------------------------------------------
elements, fractions = [], []

for comp in tqdm(df["composition"], desc="Processing compositions"):
    el, fr = extract_elements_and_fractions(comp)
    elements.append(el)
    fractions.append(fr)

df["elements"] = elements
df["fractions"] = fractions

physics_features = []
for comp in tqdm(df["composition"], desc="Adding physics features"):
    physics_features.append(composition_physics_features(comp))

physics_df = pd.DataFrame(physics_features)
df = pd.concat([df, physics_df], axis=1)

# -------------------------------------------------------------------
# STEP 4: FINAL CLEAN
# -------------------------------------------------------------------
df = df.dropna().reset_index(drop=True)

# Final column order
df = df[
    [
        "composition",
        "elements",
        "fractions",
        "volume_per_atom",
        "mean_atomic_radius",
        "std_atomic_radius",
        "mean_electronegativity",
        "mean_valence_electrons",
        "mean_atomic_mass",
    ]
]

# -------------------------------------------------------------------
# STEP 5: SAVE
# -------------------------------------------------------------------
df.to_csv("mp_volume_per_atom_clean.csv", index=False)
print("✅ Saved mp_volume_per_atom_clean.csv")
print(df.head())


Retrieving SummaryDoc documents:   0%|          | 0/153620 [00:00<?, ?it/s]

Fetching MP data: 100%|██████████| 153620/153620 [00:08<00:00, 17964.23it/s]


Raw dataset: (151156, 5)
After composition deduplication: (101819, 2)


Adding physics features: 100%|██████████| 101819/101819 [00:11<00:00, 9108.61it/s]


✅ Saved mp_volume_per_atom_clean.csv
  composition      elements          fractions  volume_per_atom  \
0          Ac          [Ac]              [1.0]        45.991230   
1     Ac2AgIr  [Ac, Ag, Ir]  [0.5, 0.25, 0.25]        27.539342   
2     Ac2AgPb  [Ac, Ag, Pb]  [0.5, 0.25, 0.25]        33.640774   
3     Ac2Br2O   [Ac, Br, O]    [0.4, 0.4, 0.2]        59.547029   
4     Ac2CdGa  [Ac, Cd, Ga]  [0.5, 0.25, 0.25]        32.036228   

   mean_atomic_radius  std_atomic_radius  mean_electronegativity  \
0              1.9500           0.000000                  1.1000   
1              1.7125           0.253414                  1.5825   
2              1.8250           0.143614                  1.6150   
3              1.3600           0.521920                  2.3120   
4              1.6875           0.276981                  1.4250   

   mean_valence_electrons  mean_atomic_mass  
0                    3.00         227.00000  
1                    6.50         188.52130  
2            